# Multi Phase Training
This notebook demonstrates training in phases, where each phase is persisted as a checkpoint.

## A. Setup

### Import configurations

In [ ]:
from src.config import ModelConfig, TokenizerConfig, TrainingConfig 

TRAINING_EPOCHS = 5 # will be used for all phases so training will proceed 5 epochs at a time (per checkpoint)

model_config = ModelConfig()
tokenizer_config = TokenizerConfig()                                                                                   
training_config = TrainingConfig(epochs=TRAINING_EPOCHS)

### Initialize untrained model

In [ ]:
from src.model.model import NanoLLM

nano_model = NanoLLM(model_config)

### Load Data

In [ ]:
from src.data.loader import load_text_from_file, preprocess_data
from src.paths import DEFAULT_DATA_FILE

stories = load_text_from_file(DEFAULT_DATA_FILE, delimiter=tokenizer_config.delimiter)

dataloader, batches_per_epoch = preprocess_data(                                                                  
      stories,                                                                                                      
      batch_size=training_config.batch_size,
      maxlen=model_config.maxlen,                                                                                   
      tokenizer_config=tokenizer_config,                                                                          
      shuffle=training_config.shuffle,                                                                              
      seed=training_config.seed,                                                                                    
)

## B. Train model with 5 epochs

In [ ]:
from src.checkpoint import default_checkpoint_path
from src.training.trainer import Trainer

# capture reference to checkpoint destination because we use it later to retreive the checkpoint
checkpoint_path_1 = default_checkpoint_path()

trainer_1 = Trainer(                                                                                                
    model=nano_model,   
    tokenizer_config=tokenizer_config,   # [optional] module falls back to default
    dataloader=dataloader,
    batches_per_epoch=batches_per_epoch,
    training_config=training_config,     # 5-epoch config
    previous_epochs_completed=0,         # no previous training
    checkpoint_path=checkpoint_path_1,   # [optional] module falls back to default
)

# Optional variables are passed explicitly in this example because we reuse those variables later.

# RUN TRAINING (includes persistence of checkpoint bundle)
# This will take a while to run depending on number of epochs.
trainer_1.train()   

In [ ]:
# Confirm that checkpoint metadata correctly reflects actual epochs trained (5)

from src.checkpoint import load_metadata

metadata_1 = load_metadata(checkpoint_path_1)
epochs_phase_1 = metadata_1.cumulative_epochs_completed
print(f"Checkpoint 1 cumulative_epochs_completed: {epochs_phase_1} (expected: {TRAINING_EPOCHS})")
assert epochs_phase_1  == TRAINING_EPOCHS 

In [ ]:
from src.compare import state_snapshot

# Capture deep copy of state after first round of training
initial_state = state_snapshot(nano_model)

## C. Reload from checkpoint

In [ ]:
import logging
from src.checkpoint import apply_checkpoint

# Create skeleton for the model
reloaded_model = NanoLLM(model_config)

# Reload weights using persisted checkpoint
#    logging lines here mute INFO-level logs (lots of orbax/checkpoint noise) while leaving exceptions and actual errors visible
logging.disable(logging.INFO)                                                                                                                                                                                                                    
apply_checkpoint(reloaded_model, checkpoint_path_1)
logging.disable(logging.NOTSET)                    

## D. Train model with 5 more epochs (total of 10)

In [ ]:
checkpoint_path_2 = default_checkpoint_path()                                                     

# Specify previous_epochs_completed so that the resulting metadata is recorded correctly.
trainer_2 = Trainer(                                                                                                
      model=reloaded_model,      
      tokenizer_config=tokenizer_config,  
      dataloader=dataloader,                                                                                        
      batches_per_epoch=batches_per_epoch,
      training_config=training_config,          # same 5-epoch config
      previous_epochs_completed=epochs_phase_1, # previous phase: 5 epochs
      checkpoint_path=checkpoint_path_2,        # location of resulting checkpoint
)

# RUN TRAINING
trainer_2.train() 

In [ ]:
# Confirm that checkpoint metadata correctly reflects actual epochs trained (5 + 5 = 10)

from src.checkpoint import load_metadata

metadata_2 = load_metadata(checkpoint_path_2)  
epochs_phase_2 = metadata_2.cumulative_epochs_completed
print(f"Checkpoint 2 cumulative_epochs_completed: {epochs_phase_2} (expected: {TRAINING_EPOCHS * 2})")
assert epochs_phase_2 == TRAINING_EPOCHS * 2

In [ ]:
# Capture deep copy of state after second round of training
resulting_state = state_snapshot(reloaded_model)

## E. Compare the model at both checkpoints

### Use CLI Tool
- We can use the defaults of the CLI tool to automatically select the two most recent checkpoints.
- Use the optional `--before` and `--after` flags to specify exact checkpoints.
- The CLI script will trigger two analysis: one comparing norms and one comparing state.

In [ ]:
# shell magic allows us to run the CLI script directly from the notebook
!uv run nanollm-compare 

### Use API Methods
Using the API directly, we can trigger norms and state analyses independently.

In [ ]:
from src.compare import (
    compare_norms,
    compare_states,
    format_norms_comparison,
    format_state_comparison,
)

In [ ]:
# initial_state and resulting_state are already saved to deep copies
# so we don't need to worry about state variables getting mutated by subsequent training
norms_report = compare_norms(initial_state, resulting_state)
width = 50
print(format_norms_comparison(norms_report, width))

In [ ]:
states_report = compare_states(initial_state, resulting_state)
width = 50
print(format_state_comparison(states_report, width))